In [17]:
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

DATASET_PATH = "floorball_dataset_ml_with_roster.csv"
MODEL_OUTPUT_PATH = "floorball_model_2.pkl"

TEST_SIZE_RATIO = 0.2
TARGET_COLUMN = "target_home_win"

REQUIRED_ROSTER_COLUMNS = [
    "home_roster_strength",
    "away_roster_strength"
]

In [9]:
df = pd.read_csv(DATASET_PATH)

print("Original dataset shape:", df.shape)

missing_required = [col for col in REQUIRED_ROSTER_COLUMNS if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing roster columns: {missing_required}")

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Missing target column: {TARGET_COLUMN}")

df = df.dropna(subset=REQUIRED_ROSTER_COLUMNS).copy()

print("Dataset shape after dropping missing roster rows:", df.shape)


Original dataset shape: (2082, 29)
Dataset shape after dropping missing roster rows: (1643, 29)


In [10]:
drop_columns = [
    "season",
    "season_norm",
    "roster_prev_season",
    "home_roster_missing",
    "away_roster_missing"
]

drop_columns = [col for col in drop_columns if col in df.columns]

X = df.drop(columns=[TARGET_COLUMN] + drop_columns).copy()
y = df[TARGET_COLUMN].copy()

preferred_categorical = ["competition_id", "league"]
categorical_features = [col for col in preferred_categorical if col in X.columns]
numeric_features = [col for col in X.columns if col not in categorical_features]

print("Feature columns:")
print(list(X.columns))

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Feature columns:
['competition_id', 'league', 'home_rank', 'away_rank', 'rank_diff', 'home_table_points', 'away_table_points', 'table_points_diff', 'home_played', 'away_played', 'home_goal_diff', 'away_goal_diff', 'goal_diff_diff', 'home_form_last5', 'away_form_last5', 'form_diff_last5', 'home_goals_for_avg_last5', 'home_goals_against_avg_last5', 'away_goals_for_avg_last5', 'away_goals_against_avg_last5', 'home_home_form_last5', 'away_away_form_last5', 'home_roster_strength', 'away_roster_strength', 'roster_strength_diff']

Numeric features:
['home_rank', 'away_rank', 'rank_diff', 'home_table_points', 'away_table_points', 'table_points_diff', 'home_played', 'away_played', 'home_goal_diff', 'away_goal_diff', 'goal_diff_diff', 'home_form_last5', 'away_form_last5', 'form_diff_last5', 'home_goals_for_avg_last5', 'home_goals_against_avg_last5', 'away_goals_for_avg_last5', 'away_goals_against_avg_last5', 'home_home_form_last5', 'away_away_form_last5', 'home_roster_strength', 'away_roster_str

In [11]:
split_index = int(len(df) * (1 - TEST_SIZE_RATIO))

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError("Train/test split failed — dataset too small.")

Train rows: 1314
Test rows: 329


In [12]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [13]:
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=10,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

models = {
    "LogisticRegression": logreg_pipeline,
    "RandomForest": rf_pipeline
}

In [15]:
results = []

for model_name, pipeline in models.items():
    print(f"Training: {model_name}")

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    print(f"Accuracy: {acc:.4f}")
    print(f"ROC AUC:  {auc:.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    results.append({
        "model_name": model_name,
        "accuracy": acc,
        "roc_auc": auc,
        "pipeline": pipeline
    })

Training: LogisticRegression
Accuracy: 0.7508
ROC AUC:  0.8164

Confusion matrix:
[[101  45]
 [ 37 146]]

Classification report:
              precision    recall  f1-score   support

           0       0.73      0.69      0.71       146
           1       0.76      0.80      0.78       183

    accuracy                           0.75       329
   macro avg       0.75      0.74      0.75       329
weighted avg       0.75      0.75      0.75       329

Training: RandomForest
Accuracy: 0.7295
ROC AUC:  0.7941

Confusion matrix:
[[ 94  52]
 [ 37 146]]

Classification report:
              precision    recall  f1-score   support

           0       0.72      0.64      0.68       146
           1       0.74      0.80      0.77       183

    accuracy                           0.73       329
   macro avg       0.73      0.72      0.72       329
weighted avg       0.73      0.73      0.73       329



In [18]:
best_result = max(results, key=lambda x: x["roc_auc"])
best_model = best_result["pipeline"]

joblib.dump(best_model, MODEL_OUTPUT_PATH)

print("\nBest model:", best_result["model_name"])
print(f"Best ROC AUC: {best_result['roc_auc']:.4f}")
print(f"Saved model: {MODEL_OUTPUT_PATH}")


Best model: LogisticRegression
Best ROC AUC: 0.8164
Saved model: floorball_model_2.pkl
